# Texas SBIR/STTR Impact — Guided Analysis
## Step 01 · Meet the data

This is a **slow, explained** walk-through. Every step answers four questions:

- **Why** — the question this step helps answer
- **Who** — where the data comes from and who produced it
- **How** — what the code actually does
- **What it means** — how to read the result

**Goal of Step 01:** load the raw SBIR/STTR award data and understand *what one row is*, then
narrow it to our study population (Texas, last 10 years). Nothing fancy yet — just meeting the data.

### 1. Set up our tools and reach the data
**Why:** we need Python's data tools (pandas) and access to the data file, which lives in your
Google Drive. **How:** we mount Drive (so the notebook can read files from it) and import pandas
(the standard table/spreadsheet library) and a small text helper (`re`).

In [ ]:
import pandas as pd   # pandas = spreadsheets in code (tables called 'DataFrames')
import re              # for cleaning up company-name text later
from google.colab import drive
drive.mount('/content/drive')

# Where our data lives. This guided folder holds a copy of the award file.
FOLDER = '/content/drive/MyDrive/fast_datasets/SBIR_TX_Guided_Analysis/'
AWARD_FILE = FOLDER + 'awards_search.csv'

# Safety net: if the copy isn't visible yet, fall back to the original in fast_datasets.
import os
if not os.path.exists(AWARD_FILE):
    AWARD_FILE = '/content/drive/MyDrive/fast_datasets/awards_search_1780524328.csv'
print('Reading award data from:', AWARD_FILE)

### 2. Who made this data, and what is it?
**Who:** the **U.S. Small Business Administration (SBA)** publishes this on **SBIR.gov**. It is the
official record of every **SBIR/STTR award** the federal government has made — pulled together from
all 11 participating agencies (Defense, NIH, NASA, NSF, Energy, etc.).

**Why it's our foundation:** the whole question — *'what is the impact of SBIR/STTR in Texas?'* —
starts from *who got awards*. This file is the ground truth for that. Everything else (are they still
alive? did they win contracts? where are they?) gets attached to *these* rows later.

**How:** `pd.read_csv` loads the file into a table `df`. We print its size and column names.

In [ ]:
df = pd.read_csv(AWARD_FILE, low_memory=False)
print('Rows (awards):', f'{len(df):,}')
print('Columns:', df.shape[1])
print()
print('Column names:')
for c in df.columns:
    print('  -', c)

### 3. What is *one row*?
**Key idea:** **one row = one award action** to one company. A single company can appear on many
rows (it won several awards over the years). Columns we'll lean on the most:

| Column | What it tells us |
|---|---|
| `Company Name` | who received it |
| `Agency` / `Branch` | which federal agency funded it (DoD, HHS/NIH, NASA…) |
| `Program` | SBIR or STTR |
| `Phase` | Phase I (feasibility) or Phase II (build-out) |
| `Award Amount` | dollars for that award |
| `Award Year` | when it was awarded |
| `State` / `City` / `ZIP` | where the company is |
| `UEI` | the government's unique ID for the company (better than the name for matching) |

**How:** let's print one real award, top to bottom, so the row feels concrete.

In [ ]:
# Show a single award as a vertical list (easier to read than a wide row)
example = df.iloc[0]
for field in ['Company Name','Agency','Program','Phase','Award Amount','Award Year','City','State','UEI']:
    print(f'{field:15}: {example[field]}')

### 4. Narrow to our study population: Texas, last 10 years
**Why:** our question is about **Texas** over the **last ~10 years**, so we keep only `State == TX`
and award years **2016–2025**. (The file actually goes back to **1983** — we'll use that deeper
history later when we need to tell *truly new* companies from ones that just look new. For now we
focus on the recent decade.)

**How:** two filters. Then we count **unique companies** — but the same firm is sometimes typed
differently ('Acme Inc.' vs 'ACME INCORPORATED'), so we make a cleaned-up `name_key` to group them.

In [ ]:
# Make numeric year, then filter
df['year'] = pd.to_numeric(df['Award Year'], errors='coerce')
tx = df[(df['State'] == 'TX') & (df['year'].between(2016, 2025))].copy()

# A cleaned company key so different spellings of one firm group together
def normalize(name):
    if pd.isna(name): return ''
    s = str(name).upper().strip()
    s = re.sub(r'[.,&\-/]', ' ', s)                          # drop punctuation
    s = re.sub(r'\b(INC|LLC|LP|LTD|CORP|CORPORATION|CO|COMPANY|INCORPORATED)\b', '', s)  # drop suffixes
    return re.sub(r'\s+', ' ', s).strip()
tx['name_key'] = tx['Company Name'].apply(normalize)

print('Texas awards, 2016-2025:', f'{len(tx):,}')
print('Unique companies      :', f"{tx['name_key'].nunique():,}")
print('Total award dollars   : $', f"{pd.to_numeric(tx['Award Amount'], errors='coerce').sum():,.0f}", sep='')

### ✅ What this means
You now have the **study population** everything else builds on:

- **~3,394 awards** went to **~878 unique Texas companies** in 2016–2025,
- worth about **$1.8 billion** in federal R&D funding.

That's the answer to the most basic version of *'what is SBIR/STTR in Texas?'* — a ~$1.8B, ~880-company
R&D pipeline over the decade. Every later step is a sharper question asked of **these same rows**:

- *Step 02* — **Who funds them?** (which agencies, and how that's changed)
- *Step 03* — **Who wins?** (repeat winners vs. one-time; how concentrated the money is)
- later — **Where are they? Are they still alive? Did the funding turn into real contracts?**

> **Check your understanding:** change `2016` to `2013` in the filter and re-run — how many more awards
> appear? That difference is the awards from 2013–2015 we chose to leave out of the 10-year window.